# 📑 Hoist Payments Reports
 
## 🔗 Links
- **[BRD - Hoist Payment Reports (Actual Payments / Daily Payments / Matched Payments) - TBD]()**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket - tbd]()**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket]()**  
  A list of findings from UAT.
 
---

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()

In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

In [0]:
RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{run_date}'), CAST('{run_date}' AS DATE)) AS DATE)").collect()[0][0]

FirstWorkingDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

FirstCalendarDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

SecondWorkingDateCurrentMonth = spark.sql(f"""
    SELECT CalendarDate
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
    AND BusinessDayOfMonth = 2 
""").collect()[0][0]

FirstWorkingDateNextMonth = spark.sql(f"""
    SELECT CalendarDate
    FROM `{env}-cdh`.reference.ref_calendar 
    WHERE MONTH(PreviousMonthLastWorkingDate) = MONTH('{RefMonth}') AND YEAR(PreviousMonthLastWorkingDate) = YEAR('{RefMonth}') AND BusinessDayOfMonth = 1
""").collect()[0][0]

SecondWorkingDateNextMonth = spark.sql(f"""
    SELECT CalendarDate
    FROM `{env}-cdh`.reference.ref_calendar 
    WHERE MONTH(PreviousMonthLastWorkingDate) = MONTH('{RefMonth}') AND YEAR(PreviousMonthLastWorkingDate) = YEAR('{RefMonth}') AND BusinessDayOfMonth = 2
""").collect()[0][0]

print(f"run_date: {run_date}")
print(f"RefMonth: {RefMonth}" )
print(f"FirstWorkingDate: {FirstWorkingDate}" )
print(f"FirstCalendarDate: {FirstCalendarDate}" )
print(f"FirstWorkingDateNextMonth: {FirstWorkingDateNextMonth}" )
print(f"SecondWorkingDateCurrentMonth: {SecondWorkingDateCurrentMonth}" )
print(f"SecondWorkingDateNextMonth: {SecondWorkingDateNextMonth}" )

# MigrationDate

In [0]:
migrationDate = spark.sql(f"""
SELECT min(cast(AssignmentDate as date)) AS migrationDate,
CASE WHEN portfolioName = 'Mercury' THEN '2019-07-01'
ELSE MIN(CAST(AssignmentDate AS DATE)) END AS cutoffdate,
PortfolioName
FROM `{env}-cdh`.primary.portfolio
WHERE PortfolioName IN {PortfolioName}
GROUP BY PortfolioName
""")

migrationDate.createOrReplaceTempView("migrationDate")

# Reassignments

In [0]:
reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

# Dexposure

In [0]:
dexposure = spark.sql(f"""
    SELECT DISTINCT 
        dexp.PortfolioName, 
        dexp.ExposureKey, 
        CDI, 
        PartyTaxRegistrationNumber, 
        AssetSegment, 
        HasActiveSettlementFlag,
        InitialAssignmentAmount
    FROM `{env}-cdh`.derivative.get_exposures_by_business_date('{run_date}', FALSE) dexp
    LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
    WHERE sourcesystem = 'profits' AND dexp.PortfolioName IN {PortfolioName}
    AND RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
                      """)

dexposure.createOrReplaceTempView("dexposure")

# InitializationExposureInfo

In [0]:
initializationexposureinfo = spark.sql(f"""                      
select Exposurekey, InitialAssignmentBalance 
from `{env}-cdh`.reference.initializationexposureinfo
"""
)
initializationexposureinfo.createOrReplaceTempView("initializationexposureinfo")

# Collections

In [0]:
Collections = spark.sql(f"""
SELECT DISTINCT c.PortfolioName, 
CASE WHEN LEFT(c.ExposureKey,3) = P.Prefix THEN c.ExposureKey ELSE CONCAT(P.Prefix,c.ExposureKey) END ExposureKey,
c.dCollectionKey as PUID,
dCollectionId as CollectionId,
ActualDate as PaymentDate, 
Amount as PaymentAmount,
MatchedDate,
RecordDate,
CASE WHEN ActualDate BETWEEN '{FirstCalendarDate}' AND '{run_date}' 
        AND NOT (ActualDate < '{FirstWorkingDate}' and MatchedDate IN ('{FirstWorkingDate}', '{SecondWorkingDateCurrentMonth}'))
        AND FinalReason = 'Matched' 
        AND CAST(CASE WHEN MatchedDate IS NULL AND FinalReason = 'Matched' THEN '{run_date}' ELSE MatchedDate END AS DATE) BETWEEN '{FirstCalendarDate}' and '{SecondWorkingDateNextMonth}' 
        THEN 1 
        ELSE 0 
        END AS IsMonthlyPayment,
CASE WHEN ActualDate <= '{run_date}' 
        AND FinalReason = 'Matched' 
        AND MatchedDate <= '{SecondWorkingDateNextMonth}' THEN 1 
        ELSE 0 
        END AS IsTotalPayment,

CASE WHEN ActualDate < '{FirstCalendarDate}'
        AND MatchedDate BETWEEN '{FirstCalendarDate}' and '{run_date}' 
        AND FinalReason = 'Matched'
        THEN 1 
        ELSE 0 
        END AS IsMatchedPayment,
CASE WHEN ActualDate BETWEEN '{FirstCalendarDate}' AND '{run_date}' 
        AND FinalReason = 'Matched' 
        THEN 1 
        ELSE 0 
        END AS IsDailyPayment,
AllocationChannel as Channel0, 
TRIM(
    REPLACE(
        REPLACE(
            REPLACE(
                REPLACE(
                    REPLACE(
                        AllocationChannel,
                        'Mercury', ''),
                    'Cell', ''),
                'Orbit', ''),
            'Light', ''),
        'Pearl', '')
) AS Channel, 
AllocationExternalCompany, 
AllocationTeam 
FROM `{env}-cdh`.derivative.dcollection c 
LEFT JOIN `{env}-cdh`.primary.portfolio as p on c.PortfolioName = p.PortfolioName AND p.BusinessDate='{run_date}'
WHERE 1=1
AND c.BusinessDate = '{SecondWorkingDateNextMonth}'
AND c.PortfolioName IN {PortfolioName}
""")
Collections.createOrReplaceTempView("Collections")

# Party

In [0]:
party = spark.sql(
f'''
Select PortfolioName, CDI, taxRegistrationNumber as PartyTaxRegistrationNumber
, CASE WHEN (Segment LIKE '%3869%' 
							AND NOT (Segment LIKE '%Rej. L.3869%')
							AND NOT (Segment LIKE '%L3869 Court Rejection%')
       				AND NOT (Segment LIKE '%L3869 Rej%')
           ) 
					 THEN 1 
					 ELSE 0
				  END AS L3869Flag, 
      
CASE WHEN Segment IS NOT NULL AND (Segment like '%-B%' OR Segment like '%-Β%' ) THEN 'Tranche B'
			      WHEN Segment IS NOT  NULL THEN 'Tranche A'											
			 END AS TrancheSegment
from 
(
    SELECT PortfolioName, CDI, TaxRegistrationNumber,Segment
          , RANK() OVER (PARTITION BY CDI ORDER BY CASE WHEN Segment IS NULL THEN 1 ELSE 0 END ASC, BusinessDate DESC) as Rank
    FROM `{env}-cdh`.derivative.dparty
    WHERE PortfolioName IN {PortfolioName}
    AND BusinessDate <= '{SecondWorkingDateNextMonth}'
    AND Segment IS NOT NULL 

) WHERE Rank = 1
'''
)
party.createOrReplaceTempView("party")

# Underlaw

In [0]:
dunderlaw = spark.sql(
f'''
SELECT CDI, CASE WHEN InLawFlag = 1 THEN 1 ELSE 0 END AS L3869Flag
FROM (
    SELECT 
            CDI,
            InLawFlag,
           UnderLawType,
            ROW_NUMBER() OVER (
                    PARTITION BY CDI
                    ORDER BY BusinessDate DESC
            ) AS rn
    FROM `{env}-cdh`.derivative.dunderlaw
    WHERE BusinessDate <= '{run_date}'
    AND PortfolioName IN {PortfolioName}
    AND UnderLawType LIKE '%3869%'
    AND SourceSystem = 'process'
) t
WHERE rn = 1
'''
)
dunderlaw.createOrReplaceTempView("dunderlaw")

# SettlementExposure

In [0]:
settlementexposure = spark.sql(
f'''
select PortfolioName, ExposureKey, SettlementStatus 
from `{env}-cdh`.primary.settlementexposure 
where BusinessDate = '{run_date}'
and PortfolioName IN {PortfolioName}
and SettlementStatus in 
('IMPLEMENTED', 'PENDING FOR IMPLEMENTATION', 'Implemented','Partially Fulfilled','Pending Implementation', 'Implemented (Inactive SOR MGR)' ,'Pending Implementation (Inactive SOR MGR)' ,'Running' ,'Working (Inactive SOR MGR)' ,'approved' ,'running' )
'''
)
settlementexposure.createOrReplaceTempView("settlementexposure")

# TotalPayments

In [0]:
TotalPaymentsPerAccount = spark.sql(
f'''
SELECT Exposurekey, 
SUM(CASE WHEN c.PaymentDate < md.cutoffdate THEN c.PaymentAmount ELSE 0 END) AS TotalPaymentsBeforeMigration
FROM Collections c
LEFT JOIN MigrationDate md 
ON c.PortfolioName = md.PortfolioName
WHERE c.IsTotalPayment = 1
GROUP BY Exposurekey
'''
)
TotalPaymentsPerAccount.createOrReplaceTempView("TotalPaymentsPerAccount")

# TotalPaymentPerMonth

In [0]:
TotalPaymentsPerMonth = spark.sql(
f'''
SELECT Exposurekey, 
SUM(CASE WHEN c.PaymentDate >= md.cutoffdate THEN c.PaymentAmount ELSE 0 END) AS TotalMonthlyPaymentsAfterMigration,
CASE WHEN c.PaymentDate >= md.cutoffdate THEN MONTH(c.PaymentDate) ELSE NULL END as InstallmentMonth,
CASE WHEN c.PaymentDate >= md.cutoffdate THEN YEAR(c.PaymentDate) ELSE NULL END as InstallmentYear
FROM Collections c
LEFT JOIN MigrationDate md 
ON c.PortfolioName = md.PortfolioName
WHERE c.IsTotalPayment = 1
GROUP BY Exposurekey, 
CASE WHEN c.PaymentDate >= md.cutoffdate THEN MONTH(c.PaymentDate) ELSE NULL END, 
CASE WHEN c.PaymentDate >= md.cutoffdate THEN YEAR(c.PaymentDate) ELSE NULL END

'''
)
TotalPaymentsPerMonth.createOrReplaceTempView("TotalPaymentsPerMonth")

# FinalTable - Payments

In [0]:
Payments_v0 = spark.sql(
f'''

SELECT DISTINCT 
c.PortfolioName,
'{run_date}' as BusinessDate,
c.Exposurekey,
c.PaymentAmount,
c.PaymentDate,
NULL as Channel0,
NULL as Team, 
c.Channel,
NULL as ExternalCompany,
ifnull(d.L3869Flag, 0) as L3869Flag,
p.TrancheSegment,
c.PUID, 
c.CollectionId as PaymentId,
de.AssetSegment, 
'Monthly' as FrequencyFlag,
cast(coalesce(MatchedDate, RecordDate) as date) as MatchedDate,
IsMonthlyPayment,
IsMatchedPayment,
CASE WHEN se.Exposurekey IS NOT NULL THEN TRUE ELSE FALSE END AS HasActiveSettlementFlag
FROM Collections c 
INNER JOIN dexposure de on c.ExposureKey = de.ExposureKey
LEFT JOIN SettlementExposure se on regexp_replace(de.ExposureKey, '[^0-9]', '') = regexp_replace(se.ExposureKey, '[^0-9]', '') and de.PortfolioName = se.PortfolioName
LEFT JOIN party p on de.CDI = p.CDI
LEFT JOIN dunderlaw d on de.CDI = d.CDI 
WHERE c.IsMonthlyPayment = 1

UNION 

SELECT DISTINCT 
c.PortfolioName,
'{run_date}' as BusinessDate,
c.Exposurekey,
c.PaymentAmount,
c.PaymentDate,
c.Channel0 as Channel0, 
c.AllocationTeam as Team, 
c.Channel,
c.allocationExternalCompany as ExternalCompany, 
ifnull(d.L3869Flag, 0) as L3869Flag,
p.TrancheSegment,
c.PUID, 
c.CollectionId as PaymentId,
de.AssetSegment, 
'Daily' as FrequencyFlag,
cast(coalesce(MatchedDate, RecordDate) as date) as MatchedDate,
IsMonthlyPayment,
IsMatchedPayment,
CASE WHEN se.Exposurekey IS NOT NULL THEN TRUE ELSE FALSE END AS HasActiveSettlementFlag
FROM Collections c 
INNER JOIN dexposure de on c.ExposureKey = de.ExposureKey
LEFT JOIN SettlementExposure se on regexp_replace(de.ExposureKey, '[^0-9]', '') = regexp_replace(se.ExposureKey, '[^0-9]', '') and de.PortfolioName = se.PortfolioName
LEFT JOIN party p on de.CDI = p.CDI
LEFT JOIN dunderlaw d on de.CDI = d.CDI 
WHERE c.IsDailyPayment = 1 or c.isMatchedPayment = 1 
'''
)
Payments_v0.createOrReplaceTempView("payments_v0")

In [0]:
Payments = spark.sql(
f'''
SELECT  
CAST(PortfolioName AS STRING),
CAST(BusinessDate AS DATE),
CAST(Exposurekey AS STRING) AS ExposureKey,
CAST(PaymentAmount AS DECIMAL(19,2)),
CAST(PaymentDate AS DATE),
CAST(Channel0 AS STRING),
CAST(Team AS STRING), 
CAST(Channel AS STRING),
CAST(ExternalCompany AS STRING),
CAST(L3869Flag AS INT),
CAST(TrancheSegment AS STRING),
CAST(PUID AS STRING), 
CAST(PaymentId AS STRING),
CAST(AssetSegment AS STRING), 
CAST(FrequencyFlag AS STRING),
cast(MatchedDate as date),
CAST(IsMonthlyPayment AS BOOLEAN),
CAST(IsMatchedPayment AS BOOLEAN), 
CAST(HasActiveSettlementFlag AS BOOLEAN) 
FROM payments_v0 
'''
)
Payments.createOrReplaceTempView("Payments")

# FinalTable - Actual Payments

In [0]:
ActualPayments = spark.sql(
f'''

SELECT 
CAST('{run_date}' AS DATE) as BusinessDate,
CAST(de.PortfolioName AS STRING),
CAST(de.Exposurekey  AS STRING) as ExposureKey,
CAST(de.CDI AS STRING),  
CAST(coalesce(CASE WHEN p.PartyTaxRegistrationNumber = '000000000' THEN NULL ELSE p.PartyTaxRegistrationNumber END, de.cdi) AS STRING) as BorrowerId, 
cast(cal.LastDateOfMonth as date) as PaymentDate, 
CAST(coalesce(de.InitialAssignmentAmount, iex.InitialAssignmentBalance, 0) AS DECIMAL(19,2)) as TotalFundedBalance, 
CAST(ifnull(tp.TotalPaymentsBeforeMigration, 0) AS DECIMAL(19,2)) as TotalPaymentsBeforeMigration,
CAST(ifnull(tppm.TotalMonthlyPaymentsAfterMigration, 0) AS DECIMAL(19,2)) as TotalMonthlyPaymentsAfterMigration,
CAST(de.AssetSegment AS STRING) as LoanType, 
CAST(ifnull(d.L3869Flag, 0) AS INT) as L3869Flag,
CAST(p.TrancheSegment AS STRING)
FROM  dexposure de
INNER JOIN party p on de.CDI = p.CDI
LEFT JOIN dunderlaw d on de.CDI = d.CDI
LEFT JOIN initializationexposureinfo IEX on iex.Exposurekey = de.Exposurekey 
LEFT JOIN TotalPaymentsPerAccount tp on de.Exposurekey = tp.Exposurekey
LEFT JOIN TotalPaymentsPerMonth tppm on de.Exposurekey = tppm.Exposurekey
LEFT JOIN (select distinct LastDateOfMonth, CalendarMonth, CalendarYear from `{env}-cdh`.reference.ref_calendar ) cal on tppm.InstallmentMonth = cal.CalendarMonth and tppm.InstallmentYear = cal.CalendarYear
'''
)
ActualPayments.createOrReplaceTempView("actualpayments")

#### Append Table Hoist_Payments

In [0]:
target_table_path_HoistPayments = f"`{env}-cdh-delivery`.investorreports.hoist_Payments"

spark.sql(f"""DELETE FROM {target_table_path_HoistPayments} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Payments = Payments.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Payments.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_HoistPayments) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_HoistPayments} SYNC METADATA")

#### Append Table Hoist_ActualPayments

In [0]:
target_table_path_HoistActualPayments = f"`{env}-cdh-delivery`.investorreports.hoist_ActualPayments"

spark.sql(f"""DELETE FROM {target_table_path_HoistActualPayments} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

ActualPayments = ActualPayments.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

ActualPayments.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_HoistActualPayments) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_HoistActualPayments} SYNC METADATA")